# 5. Interactive Visualisation with Plotly

Matplotlib (Notebook 4) produces static images — perfect for a printed
report, but you can't zoom, rotate, or hover over a point to check its exact
value. Plotly generates interactive, browser-based figures instead: ideal for
exploring multidimensional data, sharing with collaborators, or embedding in
reports, especially once you have more than 2-3 variables to look at
simultaneously (e.g. a 3-D DoE response surface you want to rotate and
inspect from different angles).

**Topics**
1. Scatter and line plots with `plotly.express`
2. Box and violin plots
3. 3-D scatter and surface plots
4. Subplots with `make_subplots`
5. Colour scales and continuous colour mapping
6. Case study: interactive DoE design space explorer

In [ ]:
import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Emit plain HTML output (not the notebook-widget MIME type), so figures
# render correctly in the built Jupyter Book pages, not just live Jupyter.
pio.renderers.default = "notebook_connected"

rng = np.random.default_rng(7)
print('Plotly version:', plotly.__version__, 'OK')

## 5.1 Scatter Plots with `plotly.express`

`plotly.express` (px) provides high-level, one-liner chart functions similar to Seaborn.

In [ ]:
# Build a polymer dataset
n = 120
grades = rng.choice(['HDPE', 'LDPE', 'PP', 'PA6', 'PEEK'], n)
grade_props = {
    'HDPE': (30, 1000, 0.96, 0),  'LDPE': (12, 300, 0.92, 0),
    'PP':   (35, 1200, 0.91, 0),  'PA6':  (75, 2800, 1.13, 1),
    'PEEK': (100, 3600, 1.32, 2),
}  # tensile_MPa, E_modulus_MPa, density, category

tensile = np.array([grade_props[g][0] + rng.normal(0, 4)  for g in grades])
E_mod   = np.array([grade_props[g][1] + rng.normal(0, 60) for g in grades])
density = np.array([grade_props[g][2] + rng.normal(0, 0.01) for g in grades])
Tg_C    = rng.normal(60, 15, n)  # generic Tg placeholder

df_poly = pd.DataFrame({
    'Grade': grades, 'Tensile_MPa': tensile, 'E_Modulus_MPa': E_mod,
    'Density_g_cm3': density, 'Tg_C': Tg_C
})

fig = px.scatter(
    df_poly,
    x='E_Modulus_MPa', y='Tensile_MPa',
    color='Grade', size='Density_g_cm3',
    hover_data=['Tg_C'],
    title='Polymer Tensile Strength vs Stiffness',
    labels={'E_Modulus_MPa': 'E Modulus (MPa)', 'Tensile_MPa': 'Tensile Strength (MPa)'},
    template='plotly_white',
)
fig.show()

## 5.2 Line Plots — Cycling Data

In [ ]:
cycles = np.arange(1, 201)
chemistries = {'LFP': 0.0005, 'NMC-622': 0.0015, 'NCA': 0.0025, 'LCO': 0.0035}

rows = []
for chem, k in chemistries.items():
    cap = 170 * np.exp(-k * cycles) + rng.normal(0, 0.3, len(cycles))
    rows.extend({'Cycle': c, 'Capacity_mAh_g': v, 'Chemistry': chem}
                for c, v in zip(cycles, cap))

df_cyc = pd.DataFrame(rows)

fig = px.line(
    df_cyc, x='Cycle', y='Capacity_mAh_g', color='Chemistry',
    title='Capacity Retention vs Cycle Number',
    labels={'Capacity_mAh_g': 'Capacity (mAh/g)'},
    template='plotly_white',
)
fig.update_traces(line=dict(width=1.5))
fig.show()

## 5.3 Box and Violin Plots

In [ ]:
# Particle size from 4 synthesis methods
methods = {'Co-precipitation': (12, 3), 'Sol-gel': (25, 6),
           'Hydrothermal': (8, 2),  'Solid-state': (80, 20)}
rows = []
for method, (mu, sigma) in methods.items():
    sizes = rng.lognormal(np.log(mu), sigma/mu, 40)
    rows.extend({'Method': method, 'Particle_nm': s} for s in sizes)
df_ps = pd.DataFrame(rows)

fig = px.violin(
    df_ps, x='Method', y='Particle_nm',
    box=True, points='outliers',
    color='Method',
    title='Particle Size Distribution by Synthesis Method',
    labels={'Particle_nm': 'Particle size (nm)'},
    template='plotly_white',
)
fig.show()

## 5.4 3-D Scatter and Surface Plots

A response that depends on two factors (like the DoE examples in Part V)
naturally forms a surface in 3-D — the two factors as the x/y ground plane,
and the response as height. A **3-D scatter plot** shows individual measured
points on this landscape; a **surface plot** shows a continuous, smoothly
interpolated version — usually a *fitted model's prediction* rather than raw
data (you rarely have measurements densely enough to draw a continuous
surface directly). Being able to rotate these plots interactively is often
the fastest way to build intuition for where a response surface peaks,
before doing any formal optimisation.

3-D plots are especially useful for visualising DoE response surfaces.

In [ ]:
# ── 3-D scatter: yield vs temperature and pressure ──────────────────────────
n_exp = 50
T     = rng.uniform(200, 400, n_exp)  # °C
P     = rng.uniform(1, 10, n_exp)     # bar
yield_pct = -0.002*T**2 + 1.4*T - 0.5*P**2 + 3*P - 60 + rng.normal(0, 2, n_exp)

fig_3d = px.scatter_3d(
    x=T, y=P, z=yield_pct,
    color=yield_pct,
    color_continuous_scale='Viridis',
    labels={'x': 'Temperature (°C)', 'y': 'Pressure (bar)', 'z': 'Yield (%)'},
    title='3-D Design Space: Reaction Yield',
    template='plotly_white',
)
fig_3d.update_traces(marker=dict(size=5))
fig_3d.show()

In [ ]:
# ── Response surface (mesh) ──────────────────────────────────────────────────
T_grid = np.linspace(200, 400, 50)
P_grid = np.linspace(1, 10, 50)
TT, PP = np.meshgrid(T_grid, P_grid)
Y_surface = -0.002*TT**2 + 1.4*TT - 0.5*PP**2 + 3*PP - 60

fig_surf = go.Figure(data=[
    go.Surface(x=T_grid, y=P_grid, z=Y_surface,
               colorscale='Viridis', opacity=0.85,
               colorbar=dict(title='Yield (%)'))
])
fig_surf.update_layout(
    title='Response Surface: Reaction Yield',
    scene=dict(
        xaxis_title='Temperature (°C)',
        yaxis_title='Pressure (bar)',
        zaxis_title='Yield (%)',
    ),
    template='plotly_white',
    height=500,
)
fig_surf.show()

## 5.5 Subplots with `make_subplots`

In [ ]:
# Combine cycle capacity and Coulombic efficiency in 2 rows
cyc = np.arange(1, 101)
cap = 168 * np.exp(-0.0015 * cyc) + rng.normal(0, 0.2, 100)
ce  = 100 - 2 * np.exp(-0.12 * cyc) + rng.normal(0, 0.05, 100)  # %

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Discharge Capacity', 'Coulombic Efficiency'),
                    vertical_spacing=0.08)

fig.add_trace(go.Scatter(x=cyc, y=cap, mode='lines+markers',
                          line=dict(color='steelblue', width=1.5),
                          marker=dict(size=3), name='Capacity'), row=1, col=1)

fig.add_trace(go.Scatter(x=cyc, y=ce, mode='lines+markers',
                          line=dict(color='darkorange', width=1.5),
                          marker=dict(size=3), name='CE'), row=2, col=1)

fig.update_yaxes(title_text='Capacity (mAh/g)', row=1)
fig.update_yaxes(title_text='CE (%)', row=2)
fig.update_xaxes(title_text='Cycle number', row=2)
fig.update_layout(title='Battery Cycling Performance', template='plotly_white',
                  height=450, showlegend=False)
fig.show()

## 5.6 Heatmaps and Colour-Mapped Data

In [ ]:
# Correlation heatmap with Plotly
n = 80
data_dict = {
    'Capacity':    rng.normal(160, 8, n),
    'First_CE':    rng.normal(88, 2, n),
    'D50_nm':      rng.normal(12, 2, n),
    'BET_m2g':     rng.normal(15, 3, n),
    'Tap_density': rng.normal(1.8, 0.15, n),
}
df_corr = pd.DataFrame(data_dict)
# Add some correlations
df_corr['Capacity'] += 0.4 * df_corr['BET_m2g']
df_corr['First_CE'] -= 0.3 * df_corr['BET_m2g']

corr_matrix = df_corr.corr().round(2)

fig = px.imshow(
    corr_matrix,
    color_continuous_scale='RdBu_r',
    color_continuous_midpoint=0,
    text_auto=True,
    title='Pearson Correlation — Cathode Properties',
    zmin=-1, zmax=1,
    template='plotly_white',
)
fig.update_layout(width=520, height=480)
fig.show()

---
## Exercises

1. **Parallel coordinates plot**: Use `px.parallel_coordinates` with the polymer
   `df_poly` dataset to explore relationships between all numeric properties simultaneously.
   Colour by `Tensile_MPa`.

2. **Animated scatter**: Create an animated `px.scatter` of the cycling data (`df_cyc`)
   showing capacity vs. cycle for all chemistries, with an animation frame per chemistry.
   Hint: use `animation_frame='Chemistry'` — first `reset_index` and sort appropriately.

3. **Contour plot**: Use `go.Contour` or `px.density_contour` to show the yield response
   surface from section 5.4 as a 2-D contour map (temperature on x, pressure on y).